In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import time
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt

In [ ]:
data = sio.loadmat('real2d.mat')
Dn = data['real2det']

In [ ]:
s1=1
s2=1
def patch2d(A, l1=35, l2=35, s1=s1, s2=s2, mode=1):
    [n1, n2] = A.shape
    if mode == 1:
        tmp = np.mod(n1 - l1, s1)
        if tmp != 0:
            A = np.concatenate((A, np.zeros([s1 - tmp, n2])), axis=0)
        tmp = np.mod(n2 - l2, s2)
        if tmp != 0:
            A = np.concatenate((A, np.zeros([A.shape[0], s2 - tmp])), axis=1)
        [N1, N2] = A.shape
        X = []
        for i1 in range(0, N1 - l1 + 1, s1):
            for i2 in range(0, N2 - l2 + 1, s2):
                tmp = np.reshape(A[i1:i1+l1, i2:i2+l2], (l1*l2, 1), order='F')
                X.append(tmp)
        X = np.array(X)
    else:
        pass
    return X[:, :, 0]


def patch2d_inv(X, n1, n2, l1=35, l2=35, s1=s1, s2=s2, mode=1):
    if mode == 1:
        tmp1 = np.mod(n1 - l1, s1)
        tmp2 = np.mod(n2 - l2, s2)
        if tmp1 != 0 and tmp2 != 0:
            A = np.zeros([n1 + s1 - tmp1, n2 + s2 - tmp2])
            mask = np.zeros([n1 + s1 - tmp1, n2 + s2 - tmp2])
        if tmp1 != 0 and tmp2 == 0:
            A = np.zeros([n1 + s1 - tmp1, n2])
            mask = np.zeros([n1 + s1 - tmp1, n2])
        if tmp1 == 0 and tmp2 != 0:
            A = np.zeros([n1, n2 + s2 - tmp2])
            mask = np.zeros([n1, n2 + s2 - tmp2])
        if tmp1 == 0 and tmp2 == 0:
            A = np.zeros([n1, n2])
            mask = np.zeros([n1, n2])
        [N1, N2] = A.shape
        id = -1
        for i1 in range(0, N1 - l1 + 1, s1):
            for i2 in range(0, N2 - l2 + 1, s2):
                id = id + 1
                A[i1:i1+l1, i2:i2+l2] = A[i1:i1+l1, i2:i2+l2] + np.reshape(X[id, :], [l1, l2], order='F')
                mask[i1:i1+l1, i2:i2+l2] = mask[i1:i1+l1, i2:i2+l2] + np.ones([l1, l2])
        A = A / mask
        A = A[0:n1, 0:n2]
    else:
        pass
    return A


In [ ]:
import scipy
[nz,nx]=Dn.shape
w1=35
w2=35
X_noisy = patch2d(Dn, w1, w2, s1, s2)
print(f"Noisy patches shape: {X_noisy.shape}")
print(f"Noisy dtype: {X_noisy.dtype}")

# Convert to float32
X_noisy = X_noisy.astype(np.float32)

Pdata_noisy = torch.from_numpy(X_noisy)
print(Pdata_noisy.dtype)

# Data split: 80% training data, 20% validation data
train_size = int(len(Pdata_noisy) * 0.8)

train_noisy = Pdata_noisy[:train_size]
valid_noisy = Pdata_noisy[train_size:]

# Print the shapes of the training set and validation set
print(f"Train noisy shape: {train_noisy.shape}")
print(f"Valid noisy shape: {valid_noisy.shape}")

# Create TensorDataset
train_data = TensorDataset(train_noisy)
valid_data = TensorDataset(valid_noisy)

batch_size1 = 64

# Create DataLoader
train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size1, shuffle=True)
valid_loader = torch.utils.data.DataLoader(valid_data, batch_size=batch_size1, shuffle=False)

print(f"Train data loader batches: {len(train_loader)}")
print(f"Valid data loader batches: {len(valid_loader)}")
print(f"Valid noisy dtype: {valid_noisy.dtype}")

In [ ]:
class FCB(nn.Module):
    def __init__(self, input_size, output_size, dropout=0.1):
        super().__init__()
        self.fc = nn.Linear(input_size, output_size)
        self.activation = nn.LeakyReLU(0.1, inplace=True)
        self.norm = nn.LayerNorm(output_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc(x)
        x = self.activation(x)
        x = self.norm(x)
        x = self.dropout(x)
        return x

In [ ]:
class SparseAttention(nn.Module):
    def __init__(self, dim, num_heads=4, window_size=2):
        super().__init__()
        assert dim % num_heads == 0, "dim must be divisible by num_heads"
        
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.window_size = window_size

        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        self.proj = nn.Linear(dim, dim)


    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale

        block_size = self.window_size * 2
        num_blocks = max(1, N // block_size)  
        mask = torch.zeros(N, N, device=x.device)
        for i in range(num_blocks):
            start = i * block_size
            end = min(start + block_size, N)  # Prevent out-of-bounds errors
            mask[start:end, start:end] = 1
        mask = mask.unsqueeze(0).unsqueeze(0).bool()

        attn = attn.masked_fill(~mask, -1e9)
        attn = attn.softmax(dim=-1)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x


class Sparse_PatchAE(nn.Module):
    def __init__(self, input_dim, embed_dim=256, depth=1, num_heads=4, window_size=2):
        super().__init__()
        self.input_dim = input_dim
        self.embed_dim = embed_dim

        self.embed = nn.Linear(input_dim, embed_dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, 2, embed_dim) * 0.02)

        self.num_heads = num_heads
        self.window_size = window_size

        self.blocks = nn.ModuleList([
            self._make_block(embed_dim) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.decoder = nn.Linear(embed_dim, input_dim)
        

    def _make_block(self, dim):
        return nn.ModuleList([
            nn.LayerNorm(dim),
            SparseAttention(dim, self.num_heads, self.window_size),
            nn.LayerNorm(dim),
            nn.Sequential(
                nn.Linear(dim, dim * 4),
                nn.GELU(),
                nn.Linear(dim * 4, dim)
            )
        ])

    def forward(self, x):
        B = x.shape[0]
        x = self.embed(x).unsqueeze(1)  # [B, 1, embed_dim]

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed

        for norm1, attn, norm2, mlp in self.blocks:
            x = x + attn(norm1(x))
            x = x + mlp(norm2(x))

        x = self.norm(x)
        patch_token = x[:, 1, :]
        recon = self.decoder(patch_token)
        return recon

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_size, dropout=0.1):
        super().__init__()
        self.fcb1 = FCB(input_size, 512, dropout)
        self.Sparse1 = Sparse_PatchAE(512, 512, depth=4)
        
        self.fcb2 = FCB(512, 128, dropout)
        self.Sparse2 = Sparse_PatchAE(128, 128, depth=4)
        
        self.fcb3 = FCB(128, 64, dropout)
        self.Sparse3 = Sparse_PatchAE(64, 64, depth=4)
        
        self.fcb4 = FCB(64, 32, dropout)
        self.Sparse4 = Sparse_PatchAE(32, 32, depth=4)
        
        self.fcb5 = FCB(32, 8, dropout)
        self.Sparse5 = Sparse_PatchAE(8, 8, depth=4)

    def forward(self, x):
        x1 = self.fcb1(x)
        y1 = self.svit1(x1)
        x2 = self.fcb2(x1)
        y2 = self.svit2(x2)
        x3 = self.fcb3(x2)
        y3 = self.svit3(x3)
        x4 = self.fcb4(x3)
        y4 = self.svit4(x4)
        x5 = self.fcb5(x4)
        y5 = self.svit5(x5)
        return x1, x2, x3, x4, x5, y1, y2, y3, y4, y5

In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_size, dropout=0.1):
        super().__init__()
        self.fcb5 = FCB(8 + 8, 32, dropout)
        self.fcb4 = FCB(32 + 32, 64, dropout)
        self.fcb3 = FCB(64 + 64, 128, dropout)
        self.fcb2 = FCB(128 + 128, 512, dropout)
        self.fcb1 = FCB(512 + 512, output_size, dropout)

    def forward(self, x1, x2, x3, x4, x5, y1, y2, y3, y4, y5):
        y51 = torch.cat([x5, y5], dim=1)
        y41 = self.fcb5(y51)
        y41 = torch.cat([y41, y4], dim=1)
        y31 = self.fcb4(y41)
        y31 = torch.cat([y31, y3], dim=1)
        y21 = self.fcb3(y31)
        y21 = torch.cat([y21, y2], dim=1)
        y11 = self.fcb2(y21)
        y11 = torch.cat([y11, y1], dim=1)
        output = self.fcb1(y11)
        return output

In [ ]:
class AutoCoder(nn.Module):
    def __init__(self, input_size, output_size, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(input_size, dropout)
        self.decoder = Decoder(output_size, dropout)

    def forward(self, x):
        x1, x2, x3, x4, x5, y1, y2, y3, y4, y5 = self.encoder(x)
        output = self.decoder(x1, x2, x3, x4, x5, y1, y2, y3, y4, y5)
        return output

In [ ]:
input_dim=w1*w2
model = AutoCoder(input_size=input_dim, output_size=input_dim)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

In [ ]:
def huber_loss(pred, target, delta=1.3):
    error = pred - target
    abs_error = torch.abs(error)
    quadratic = torch.clamp(abs_error, max=delta)
    linear = abs_error - quadratic
    loss = 0.5 * quadratic ** 2 + delta * linear
    return loss.mean()

In [ ]:
# ======== Model initialization ========
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AutoCoder(input_size=w1*w2, output_size=w1*w2, dropout=0.1).to(device)
# Keep only pure linear layers to avoid stitching artifacts
model.decoder.fcb1 = torch.nn.Linear(1024, w1*w2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
# ======= Training loop =======
es_thres = 5
es_cnt = 0
prev_valid_loss = float('inf')
loss_train = []
loss_valid = []
num_epochs = 50
# Random masking ratio
mask_ratio = 0.35 

w1, w2 = 35, 35

print("Start Unsupervised Training (Random Blind-Spot Prediction)...")
for epoch in range(num_epochs):
    
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        noisy_patch = batch[0].to(device)  
        B = noisy_patch.shape[0]
        
       
        # Generate a [0, 1) uniform random number matrix with the same shape as the input
        rand_tensor = torch.rand_like(noisy_patch)
        
        # Generate boolean mask: set positions smaller than mask_ratio to True (pixels to be masked)
        # The shape of random_mask_2d is also [B, w1*w2]
        random_mask_2d = rand_tensor < mask_ratio
        
        # Construct input: invert the mask positions bitwise (~) to set masked positions to 0 and keep original values for others
        input_patch = noisy_patch * (~random_mask_2d)
        target_patch = noisy_patch
        # ------------------------------------------------
        
        optimizer.zero_grad()
        outputs = model(input_patch)
        
         # -------- Loss calculation: only focus on masked scatter points --------
        # Use boolean indexing outputs[random_mask_2d] to directly extract a 1D tensor at True positions
        # Calculate Huber Loss only for masked points, which is more efficient than global multiplication
        rec_loss = huber_loss(outputs[random_mask_2d], target_patch[random_mask_2d], delta=1.3)
        
      
        outputs_4d = outputs.view(B, 1, w1, w2)
        loss = rec_loss
        # ------------------------------------------------
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * input_patch.size(0)

    train_loss = train_loss / len(train_noisy)
    loss_train.append(train_loss)


    # The validation set must also use the same Random Masking logic for evaluation
    model.eval()
    valid_loss = 0.0
    with torch.no_grad():
        for batch in valid_loader:
            valid_patch = batch[0].to(device)
            B_val = valid_patch.shape[0]
            
            rand_tensor_val = torch.rand_like(valid_patch)
            random_mask_2d_val = rand_tensor_val < mask_ratio
            
            # Construct validation set input
            input_val_patch = valid_patch * (~random_mask_2d_val)
            
            outputs = model(input_val_patch)
            
            # Calculate validation loss
            rec_loss_val = huber_loss(outputs[random_mask_2d_val], valid_patch[random_mask_2d_val], delta=1.3)
            
            outputs_4d_val = outputs.view(B_val, 1, w1, w2)
            val_loss = rec_loss_val
            
            valid_loss += loss.item() * valid_patch.size(0)

    valid_loss = valid_loss / len(valid_noisy)
    loss_valid.append(valid_loss)

    print(f"Epoch[{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Valid Loss: {valid_loss:.4f}")

    # ======== Early stopping  ========
    if valid_loss < prev_valid_loss:
        es_cnt = 0
        prev_valid_loss = valid_loss
        torch.save(model.state_dict(), 'real2d_ISANET_denoise.pth')
    else:
        es_cnt += 1
        
    if es_cnt >= es_thres:
        print(f"Early stopping at epoch {epoch+1}")
        break

print("Unsupervised learning weights have been saved...")

In [ ]:
#### Inference and post-processing
model = AutoCoder(input_size=w1*w2, output_size=w1*w2, dropout=0.1).to(device)
model.decoder.fcb1 = torch.nn.Linear(1024, w1*w2).to(device) # Ensure structural alignment
model.load_state_dict(torch.load('real2d_ISANET_denoise.pth'))
model.eval()

# Full-data inference
with torch.no_grad():
    Pdata_noisy = Pdata_noisy.to(device)
    outputs = model(Pdata_noisy)
    
    # Convert to numpy array for fine processing
    outputs = outputs.cpu().numpy()
    # Obtain the absolute maximum value of the entire output data as the baseline
    max_amp = np.max(np.abs(outputs))
    threshold = max_amp * 0.008  
    
    outputs = np.sign(outputs) * np.maximum(np.abs(outputs) - threshold, 0)
    
print("== Inference output statistics ===")
print(f"Total number of output patches: {outputs.shape[0]}")
print(f"Total number of output patches: {np.count_nonzero(outputs.any(axis=1))}")
print(f"Output maximum value: {outputs.max():.4f},  minimum value: {outputs.min():.4f}, mean value: {outputs.mean():.4f}")

# Inverse blocking to merge into a complete image
denoised_data = patch2d_inv(outputs, n1=nz, n2=nx, l1=35, l2=35, s1=s1, s2=s2)

# Save data
scipy.io.savemat("real2d_ISANET_result.mat", {"D_denoised": denoised_data})

print("✅ File saved：real2d_ISANET_result.mat")
print(f"✅ Denoised data shape: {denoised_data.shape}")
